# OCTA Classifier — Grid Search


In [ ]:
import sys
import logging
from pathlib import Path
import torch

PROJECT_ROOT = Path().resolve()                         
sys.path.insert(0, str(PROJECT_ROOT))

DATA_ROOT    = PROJECT_ROOT.parent / "data"              
EXCEL_PATH   = DATA_ROOT / "master_excels" / "master_table.xlsx"
ENCODER_PATH = PROJECT_ROOT.parent / "encoders" / "results" / "phase2" / "grl" / "pretrained" / "final_encoder.pth"
RESULTS      = PROJECT_ROOT / "results"

# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)

print(f"Device      : {device}")
print(f"Project     : {PROJECT_ROOT}")
print(f"Data root   : {DATA_ROOT}")
print(f"Excel       : {EXCEL_PATH}")
print(f"Encoder     : {ENCODER_PATH}")
print(f"Excel exists: {EXCEL_PATH.exists()}")
print(f"Encoder exists: {ENCODER_PATH.exists()}")

In [ ]:
from scripts.training import run_grid_search, TrainConfig

GRID = {
    "unfreeze_last_n_blocks" : [0, 2, 4],
    "lr"                     : [3e-4, 1e-4, 3e-5],
    "hidden_dims"            : [[512], [512, 256], [512, 512, 256], [256, 128]],
    "dropout"                : [0.3, 0.4, 0.5],
    "label_smoothing"        : [0.0, 0.05, 0.10],
    "encoder_mode"           : ["cls", "cls_mean", "mean_patch"],
    "modality_dropout_prob"  : [0.0, 0.2, 0.3, 0.4],
}

N_RUNS = 300

BASE_CFG = TrainConfig(
    epochs        = 60,
    batch_size    = 32,
    weight_decay  = 1e-4,
    warmup_epochs = 10,
    min_lr        = 1e-6,
    grad_clip     = 1.0,
    use_amp       = True,
    class_weights = "auto",
    encoder_lr_multiplier = 0.05,
    use_bn        = True,
    tf_num_heads  = 4,
    tf_num_layers = 1,
    tf_dropout    = 0.1,
    num_workers   = 4,
    seed          = 42,
)

# Všetky dostupné dáta

In [ ]:
run_grid_search(
    experiment_name = "full",
    grid_config     = GRID,
    encoder_path    = ENCODER_PATH,
    excel_path      = EXCEL_PATH,
    data_root       = DATA_ROOT,
    results_root    = RESULTS / "grid_search",
    n_runs          = N_RUNS,
    base_cfg        = BASE_CFG,
    device          = device,
)

# SVP DCP 

In [ ]:
run_grid_search(
    experiment_name = "svp_dcp",
    grid_config     = GRID,
    encoder_path    = ENCODER_PATH,
    excel_path      = EXCEL_PATH,
    data_root       = DATA_ROOT,
    results_root    = RESULTS / "grid_search",
    n_runs          = N_RUNS,
    base_cfg        = BASE_CFG,
    device          = device,
)

# SVP + DCP vyvážené

In [ ]:
run_grid_search(
    experiment_name = "svp_dcp_ballanced",
    grid_config     = GRID,
    encoder_path    = ENCODER_PATH,
    excel_path      = EXCEL_PATH,
    data_root       = DATA_ROOT,
    results_root    = RESULTS / "grid_search",
    n_runs          = N_RUNS,
    base_cfg        = BASE_CFG,
    device          = device,
)